In [1]:
import xarray as xr
import pandas as pd
import os 



base_dir = "/Users/jakobwerkgarner/code/mt_dsnow/"
os.chdir(base_dir)
from datetime import timedelta


from pathlib import Path
import subprocess
import pandas as pd
from io import StringIO



In [3]:
#######functions


def build_fit_tibble(d_obs_fit, start_of_block=8):
    tibbles = []

    for name, df in d_obs_fit.items():
        if name == "Sta.Maria":
            continue  # optional exclusion

        df = df.copy()
        tib = pd.DataFrame({
            'date': df.index,
            'name': name,
            'hs': df['hs'].values ,  
            'swe_obs': df['swe_obs'].values
        })

        # Add 'block' as seasonal year
        tib['block'] = tib['date'].apply(lambda d: d.year - 1 if d.month < start_of_block else d.year)

        tibbles.append(tib)

    return pd.concat(tibbles, ignore_index=True)




def calc_metrics(df):
    # Filter only rows with valid observed SWE values
    valid = df.dropna(subset=["swe_obs", "swe_mod"])

    # Compute RMSE
    rmse = np.sqrt(np.mean((valid["swe_mod"] - valid["swe_obs"])**2))

    # Compute Bias
    bias = np.mean(valid["swe_mod"] - valid["swe_obs"])

    print(f"RMSE: {rmse:.3f}")
    print(f"Bias: {bias:.3f}")

    return(rsme, bias)

In [4]:

# 1. Load the NetCDF dataset
ds = xr.open_dataset("calibration/calibration_data/raw_data/Mag25/SLF_dataset/Mag25_all.nc")

# 2. Rename variables for consistency
ds = ds.rename({
    "SWE": "swe_obs",
    "HS": "hs"
})

# 3. Reduce dataset to only hs and swe_obs
reduced_ds = ds[["hs", "swe_obs"]]

# 4. Loop through stations and create individual pandas DataFrames
station_dfs = {}

station_names = reduced_ds["station"].values

for station in station_names:
    print(f"Processing station: {station}")
    
    # Select the data for a single station
    ds_station = reduced_ds.sel(station=station)
    
    # Convert to pandas DataFrame
    df_station = ds_station.to_dataframe().reset_index()
    
    # Add station name as a column (in case it gets lost in reset_index)
    df_station["name"] = station
    
    # Rename time for consistency
    df_station = df_station.rename(columns={"time": "date"})
    
    # Save into dictionary
    station_dfs[station] = df_station



#experimental name = 'Adelboden'

season_start='-08-01' 
season_end='-07-31'

d_obs_fit, d_obs_val = {}, {}



for name in station_names:

    print(f"Processing site: {name}")

    site_df = station_dfs[name]
    site_df.set_index('date', inplace=True)
    years = site_df.index.year.unique()
    winters4fit, winters4val = [], []
    idx_y = 0

    for y in years:

        start = pd.to_datetime(f"{y}{season_start}")
        end = pd.to_datetime(f"{y+1}{season_end}") + timedelta(days=1)
        winter = site_df[(site_df.index >= start) & (site_df.index < end)]


        if len(winter) < 200:
            print(f"Skipping {y} ({name}): only {len(winter)} records.")

        if len(winter) < 365 and (winter['hs'].iloc[0] > 0.05 or winter['hs'].iloc[-1] > 0.05):
            print(f"Skipping {y} ({name}): edge hs > 0.05.")


        # fit with season staring with an even year
        if winter.index[idx_y].year % 2 == 0:
            winters4fit.append(winter)

        # validate with season with odd winters
        else:
            winters4val.append(winter)

    if winters4fit:
        d_obs_fit[name] = pd.concat(winters4fit)
    if winters4val:
        d_obs_val[name] = pd.concat(winters4val)



d_obs_fit_tibble = build_fit_tibble(d_obs_fit)
d_obs_val_tibble = build_fit_tibble(d_obs_val)

Processing station: Adelboden
Processing station: Gadmen
Processing station: Grindelwald_Bort
Processing station: Gsteig
Processing station: Gantrisch
Processing station: Leysin
Processing station: Muerren
Processing station: Saanenmoeser
Processing station: Wengen
Processing station: Srenberg
Processing station: Stoos
Processing station: Braunwald
Processing station: Malbun
Processing station: St_Margrethenberg
Processing station: Binn
Processing station: Bourg_St_Pierre
Processing station: Fionnay
Processing station: Grimentz
Processing station: Lauchernalp
Processing station: Montana
Processing station: Muenster
Processing station: Saas_Fee
Processing station: Simplon_Dorf
Processing station: Ulrichen
Processing station: Wiler
Processing station: Bivio
Processing station: Davos_Flueelastr
Processing station: Juf
Processing station: Obersaxen
Processing station: Pusserein
Processing station: St_Antoenien
Processing station: Sedrun
Processing station: Spluegen
Processing station: Vals

In [5]:
import pandas as pd
import subprocess
from pathlib import Path
from io import StringIO
import numpy as np

def evaluate_deltasnow_model(
    d_obs_fit_tibble: pd.DataFrame,
    r_script_path: str,
    params: dict,
    temp_dir: str = "/tmp",
    min_block_length: int = 100
):
    """
    Runs the Δ-SNOW model block-wise via R and returns evaluation metrics.
    
    Parameters:
    - d_obs_fit_tibble: DataFrame with ['date', 'name', 'hs', 'swe_obs', 'block']
    - r_script_path: Path to the minimal_delta_snow_runner.R
    - params: Dictionary with model parameters
    - temp_dir: Temporary directory for storing input CSVs
    - min_block_length: Threshold for skipping too-short seasons

    Returns:
    - merged_df: DataFrame with ['date', 'name', 'hs', 'swe_obs', 'swe_mod']
    - metrics: Dictionary with RMSE and bias
    """
    
    # Assume d_obs_fit_tibble is already prepared
    fit_df = d_obs_fit_tibble.copy()
    fit_df["date"] = pd.to_datetime(fit_df["date"])  # Ensure datetime format

    input_csv = Path(temp_dir) / "hs_input.csv"
    results = []

    stations = fit_df["name"].unique()
    blocks = fit_df["block"].unique()

    for station in stations:
        for block in blocks:
            df_sub = fit_df[(fit_df["name"] == station) & (fit_df["block"] == block)]

            if len(df_sub) < min_block_length:
                continue  # Skip too-short seasons

            # Reindex to daily
            df_sub = df_sub.sort_values("date")
            full_range = pd.date_range(df_sub["date"].min(), df_sub["date"].max(), freq="D")
            df_sub = df_sub.set_index("date").reindex(full_range).rename_axis("date").reset_index()

            df_sub["hs"] = df_sub["hs"].fillna(0)  # Fill missing snow depths

            # Write input to temp CSV (only input is written!)
            df_sub[["date", "hs"]].to_csv(input_csv, index=False)

            # Prepare Rscript call
            cmd = ["Rscript", r_script_path, "--in", str(input_csv)]
            for k, v in params.items():
                cmd += [f"--{k}", str(v)]

            result = subprocess.run(cmd, capture_output=True, text=True)

            if result.returncode != 0:
                print(f"Δ-SNOW failed for {station}, {block}:\n{result.stderr}")
                continue

            try:
                # Parse result directly from R stdout (no file I/O)
                df_result = pd.read_csv(StringIO(result.stdout), parse_dates=["date"])
                df_result["station"] = station
                df_result["block"] = block
                results.append(df_result)
                #print(f" Success: {station}, {block}")
            except Exception as e:
                print(f"Parsing error for {station}, {block}: {e}")
                continue

    # Combine all results
    if not results:
        raise RuntimeError("No results returned from Δ-SNOW.")

    all_results_df = pd.concat(results, ignore_index=True)

    # Step 1: Rename for matching
    mod_df = all_results_df.rename(columns={"station": "name"})

    # Step 2: Merge on 'date' and 'name'
    merged_df = pd.merge(
        d_obs_fit_tibble,
        mod_df[["date", "name", "swe_mod"]],
        on=["date", "name"],
        how="inner"
    )

    # Step 3: Reorder columns (optional)
    merged_df = merged_df[["date", "name", "hs", "swe_obs", "swe_mod"]]

    # Step 4: Compute metrics for available values
    valid = merged_df.dropna(subset=["swe_obs", "swe_mod"])
    rmse = np.sqrt(np.mean((valid["swe_mod"] - valid["swe_obs"]) ** 2))
    bias = np.mean(valid["swe_mod"] - valid["swe_obs"])
    metrics = {"rmse": rmse, "bias": bias}

    return merged_df, metrics

In [6]:
from scipy.optimize import minimize

def run_lbfgsb_optimization(
    d_obs_fit_tibble,
    r_script_path,
    param_names,
    param_initial,
    param_bounds,
    scale_factors=None,
    temp_dir="/tmp"
):
    """
    Runs L-BFGS-B optimization to minimize RMSE using the evaluate_deltasnow_model function.
    
    Parameters:
    - d_obs_fit_tibble: Input dataframe
    - r_script_path: R runner script path
    - param_names: List of parameter names
    - param_initial: List of initial guesses (unscaled)
    - param_bounds: List of (lower, upper) tuples (unscaled)
    - scale_factors: Optional list of scaling factors (same length)
    
    Returns:
    - result: scipy.optimize result object
    """

    scale_factors = scale_factors or [1.0] * len(param_initial)

    def objective(x_scaled):
        # Unscale parameters
        x_unscaled = [x * s for x, s in zip(x_scaled, scale_factors)]
        params_dict = dict(zip(param_names, x_unscaled))

        _, metrics = evaluate_deltasnow_model(
            d_obs_fit_tibble=d_obs_fit_tibble,
            r_script_path=r_script_path,
            params=params_dict,
            temp_dir=temp_dir
        )
        return metrics["rmse"]

    # Prepare scaled initial and bounds
    x0 = [v / s for v, s in zip(param_initial, scale_factors)]
    bounds_scaled = [(lo / s, hi / s) for (lo, hi), s in zip(param_bounds, scale_factors)]

    result = minimize(
        objective,
        x0=x0,
        method="L-BFGS-B",
        bounds=bounds_scaled,
        options={"disp": True, "maxiter": 50}
    )

    # Unscale final result
    optimized_params = [x * s for x, s in zip(result.x, scale_factors)]
    result.final_params = dict(zip(param_names, optimized_params))
    
    return result

In [7]:
# Initial parameters
param_names = ["rho.max", "rho.null", "c.ov", "k.ov", "k", "tau", "eta.null"]
param_initial = [400, 80, 0.0005, 0.25, 0.03, 0.02, 8500000]
param_bounds = [
    (300, 600),     # rho.max
    (50, 200),      # rho.null
    (0.0, 0.001),   # c.ov
    (0.01, 10),     # k.ov
    (0.01, 0.2),    # k
    (0.01, 0.2),    # tau
    (1e6, 2e7)      # eta.null
]
scale_factors = [1000, 1000, 0.001, 1, 0.01, 0.1, 1e7]

# Run optimization
result = run_lbfgsb_optimization(
    d_obs_fit_tibble=d_obs_fit_tibble,
    r_script_path="calibration/helpers/minimal_delta_snow_runner.R",
    param_names=param_names,
    param_initial=param_initial,
    param_bounds=param_bounds,
    scale_factors=scale_factors,
    temp_dir="/tmp"
)

print("Optimized parameters (unscaled):")
print(result.final_params)
print("Final RMSE:", result.fun)

Δ-SNOW failed for Adelboden, 2016:
Error in swe.delta.snow(data = df, model_opts = model_opts, layer = FALSE) : 
  'k.ov' must be < 1
Execution halted

Δ-SNOW failed for Adelboden, 2018:
Error in swe.delta.snow(data = df, model_opts = model_opts, layer = FALSE) : 
  'k.ov' must be < 1
Execution halted

Δ-SNOW failed for Adelboden, 2020:
Error in swe.delta.snow(data = df, model_opts = model_opts, layer = FALSE) : 
  'k.ov' must be < 1
Execution halted

Δ-SNOW failed for Gadmen, 2016:
Error in swe.delta.snow(data = df, model_opts = model_opts, layer = FALSE) : 
  'k.ov' must be < 1
Execution halted

Δ-SNOW failed for Gadmen, 2018:
Error in swe.delta.snow(data = df, model_opts = model_opts, layer = FALSE) : 
  'k.ov' must be < 1
Execution halted

Δ-SNOW failed for Gadmen, 2020:
Error in swe.delta.snow(data = df, model_opts = model_opts, layer = FALSE) : 
  'k.ov' must be < 1
Execution halted

Δ-SNOW failed for Grindelwald_Bort, 2016:
Error in swe.delta.snow(data = df, model_opts = model_o

RuntimeError: No results returned from Δ-SNOW.

# Funciton

In [13]:
import yaml
def load_config(config_path="config_calib.yml"):
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)
    return cfg

In [14]:
load_config()

{'paths': {'nc_file': 'calibration/calibration_data/raw_data/Mag25/SLF_dataset/Mag25_all.nc',
  'r_script': 'calibration/helpers/minimal_delta_snow_runner.R',
  'output_dir': 'outputs'},
 'optimization': {'workers': 8, 'maxiter': 40},
 'parameters': {'names': ['rho.max',
   'rho.null',
   'c.ov',
   'k.ov',
   'k',
   'tau',
   'eta.null'],
  'initial': [400, 80, 0.0005, 0.25, 0.03, 0.02, '8.5e6'],
  'bounds': [[300, 600],
   [50, 200],
   [0.0, 0.001],
   [0.01, 10],
   [0.01, 0.2],
   [0.01, 0.2],
   ['1e6', '2e7']],
  'scale': [1000, 1000, 0.001, 1, 0.01, 0.1, '1e7']}}

In [5]:
import os 

os.getcwd()



'/Users/jakobwerkgarner/code/mt_dsnow/calibration/calibration_py/L-BFGS-B'

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

def plot_station_blocks_calendar_dual_fixed(df):
    df = df[df['swe_obs'].notna()].copy()
    df['date'] = pd.to_datetime(df['date'])
    df['block'] = df['date'].apply(lambda d: d.year - 1 if d.month < 8 else d.year)

    all_stations = sorted(df['name'].unique())
    stations_left = all_stations[:21]
    stations_right = all_stations[21:]

    fig, axes = plt.subplots(1, 2, figsize=(18, 0.6 * max(len(stations_left), len(stations_right))), sharex=True)

    unique_blocks = sorted(df['block'].unique())
    palette = sns.color_palette("tab20", len(unique_blocks))
    color_map = {block: palette[i % len(palette)] for i, block in enumerate(unique_blocks)}

    for ax, stations, title in zip(axes, [stations_left, stations_right], ['Stations 1–21', 'Stations 22+']):
        station_indices = {station: i for i, station in enumerate(stations)}

        for station in stations:
            station_data = df[df['name'] == station]
            for block in station_data['block'].unique():
                block_data = station_data[station_data['block'] == block]
                min_date = block_data['date'].min()
                max_date = block_data['date'].max()
                y = station_indices[station]

                ax.hlines(
                    y=y,
                    xmin=min_date,
                    xmax=max_date,
                    color=color_map[block],
                    linewidth=10,
                    alpha=0.9
                )

        ax.set_yticks(range(len(stations)))
        ax.set_yticklabels(stations)
        ax.set_title(f'SWE Observation Periods – {title}', fontsize=13)
        ax.set_xlabel('Date')
        ax.set_ylabel('Station')
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    plt.xticks(rotation=45)
    sns.despine()
    plt.tight_layout()
    plt.show()